In [3]:
# ==============================
# 1. INSTALL
# ==============================
!pip install -qU langchain-community langchain-huggingface pymupdf faiss-cpu tqdm

# ==============================
# 2. IMPORTS
# ==============================
import os, glob, re
from tqdm import tqdm
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==============================
# 3. PATHS
# ==============================
DATASET_PATH = "/kaggle/input/datasets/vangap/indian-supreme-court-judgments/pdfs"
SAVE_PATH = "/kaggle/working/chunked_supreme_court_index"

# ==============================
# 4. CLEAN TEXT
# ==============================
def clean_text(text):
    text = re.sub(r"Page \d+ of \d+", "", text)
    text = re.sub(r"http://JUDIS\.NIC\.IN", "", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ==============================
# 5. METADATA EXTRACTION
# ==============================
def extract_case_fields(text):

    petitioner = re.search(r"PETITIONER:\s*(.*?)\s+Vs\.", text, re.IGNORECASE)
    respondent = re.search(r"RESPONDENT:\s*(.*?)\s+DATE OF JUDGMENT", text, re.IGNORECASE)
    date = re.search(r"DATE OF JUDGMENT[:\s]*(\d{2}/\d{2}/\d{4})", text)

    bench_blocks = re.findall(r"BENCH:\s*(.*?)(?=BENCH:|CITATION|ACT:|HEADNOTE|JUDGMENT)", text, re.IGNORECASE)

    bench_names = []
    for block in bench_blocks:
        for line in block.split("\n"):
            if len(line.strip()) > 3:
                bench_names.append(line.strip())

    judgment_match = re.search(
        r"(Appeal allowed|Appeal dismissed|Petition dismissed|Petition allowed|Convicted|Acquitted)",
        text[-1500:], re.IGNORECASE
    )

    return {
        "petitioner": petitioner.group(1).strip() if petitioner else "UNKNOWN",
        "respondent": respondent.group(1).strip() if respondent else "UNKNOWN",
        "date": date.group(1) if date else "UNKNOWN",
        "bench": ", ".join(set(bench_names)) if bench_names else "UNKNOWN",
        "judgment": judgment_match.group(1) if judgment_match else "UNKNOWN"
    }

# ==============================
# 6. LEGAL CHUNKING (IMPORTANT)
# ==============================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " "]  # legal-friendly splitting
)

# ==============================
# 7. BUILD KB (BATCHED)
# ==============================
def build_kb(batch_size=2000):

    pdf_files = glob.glob(os.path.join(DATASET_PATH, "*.pdf"))
    print(f"✅ Total PDFs: {len(pdf_files)}")

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={'device': 'cuda'}
    )

    existing_batches = glob.glob(f"{SAVE_PATH}_batch_*")
    start_batch = len(existing_batches)

    print(f"🔁 Resuming from batch: {start_batch}")

    total_batches = (len(pdf_files) // batch_size) + 1

    for batch_idx in range(start_batch, total_batches):

        start = batch_idx * batch_size
        end = start + batch_size
        batch_files = pdf_files[start:end]

        if not batch_files:
            break

        print(f"\n🚀 Processing Batch {batch_idx+1}/{total_batches}")

        documents = []

        for file_path in tqdm(batch_files):
            try:
                loader = PyMuPDFLoader(file_path)
                pages = loader.load()

                full_text = " ".join([p.page_content for p in pages])
                full_text = clean_text(full_text)

                fields = extract_case_fields(full_text)

                # 🔥 CHUNKING HERE
                chunks = splitter.split_text(full_text)

                for chunk in chunks:
                    doc = Document(
                        page_content=chunk,
                        metadata={
                            "source": file_path,
                            "petitioner": fields["petitioner"],
                            "respondent": fields["respondent"],
                            "date": fields["date"],
                            "bench": fields["bench"],
                            "judgment": fields["judgment"]
                        }
                    )
                    documents.append(doc)

            except:
                continue

        vector_db = FAISS.from_documents(documents, embeddings)

        batch_path = f"{SAVE_PATH}_batch_{batch_idx}"
        vector_db.save_local(batch_path)

        print(f"✅ Saved Batch {batch_idx}")

# ==============================
# 8. MERGE
# ==============================
def merge_batches():

    batch_paths = sorted(glob.glob(f"{SAVE_PATH}_batch_*"))

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )

    db = None

    for path in batch_paths:
        print(f"🔗 Loading {path}")
        new_db = FAISS.load_local(path, embeddings, allow_dangerous_deserialization=True)

        if db is None:
            db = new_db
        else:
            db.merge_from(new_db)

    db.save_local(SAVE_PATH + "_final")
    print("🚀 Final FAISS saved!")

# ==============================
# 9. RUN
# ==============================
build_kb(batch_size=2000)
merge_batches()

✅ Total PDFs: 48294


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

100%|██████████| 294/294 [00:15<00:00, 18.46it/s]


✅ Saved Batch 24


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_0
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_1
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_10
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_11
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_12
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_13
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_14
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_15
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_16
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_17
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_18
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_19
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_2
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_20
🔗 Loading /kaggle/working/chunked_supreme_court_index_batch_21
🔗 Loading /kaggle/working/chunked_supreme_court_index_batc

In [2]:
!pip install -qU langchain-text-splitters